# Tutorial 02 — Governance on `alexandria-server` (`/v1`) & Immutable Distribution (`/pub`)

In this notebook we start a real **`alexandria-server`** (`C-HUB`) instance and walk through the entire governance lifecycle using **real HTTP requests**:
1. Resolving the **W3C `did:web`** document (`GET /.well-known/did.json`).
2. Bearer authentication and the role-based authorization matrix (`editor`, `reviewer`, `releaser`, `member`).
3. Creating and updating **Drafts** with optimistic concurrency control (`ETag` / `If-Match`).
4. Proposing an **immutable Candidate (`:propose`)** that compiles the `eunomia.profiles/v1` YAML bundle on the fly.
5. Role separation and cryptographic approval of a **Release (`:approve`)**.
6. Querying the **JCS + Ed25519 signed version listing (`GET /v1/versions`)** and fetching content-addressed immutable artifacts under **`/pub/*`**.

In [1]:
from lab import AlexandriaLab, http_request

lab = AlexandriaLab(compiler="dsl", initial_clock="2026-09-01T10:00:00Z")
lab.start_server()

✔ alexandria-server listening at http://127.0.0.1:62593 (DID: did:web:127.0.0.1%3A62593)

## 1. Identity Discovery: `GET /.well-known/did.json`

On startup, `alexandria-server` generates its Ed25519 key pair and publishes its verification method at `/.well-known/did.json` (`REQ-HUB-010`, `DP-4`). This endpoint is public:

In [2]:
status, headers, did_doc = http_request("GET", f"{lab.hub_url}/.well-known/did.json")
assert status == 200
assert did_doc["id"] == lab.hub_did

cache-control,no-cache
connection,close
content-length,403
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 2. Authentication and Access Control on `/v1/*` (`REQ-HUB-017`)

All resources under `/v1/` require a valid Bearer token (`Authorization: Bearer <token>`):
- Without a token, the server responds `401 Unauthorized` with `WWW-Authenticate: Bearer realm="vhub"` and code `API_FORBIDDEN_ROLE`.
- With an insufficient role (`member-token` attempting to create a draft), it responds `403 Forbidden` (`API_FORBIDDEN_ROLE`).
- With `member-token` on `GET /v1/versions` before any version has been published, authentication succeeds and returns `404 API_NOT_FOUND` (`"no version has been published yet"`).

In [3]:
print("1) Unauthenticated request (expected 401):")
st_unauth, _, _ = http_request("GET", f"{lab.hub_url}/v1/versions")
assert st_unauth == 401

print("2) Insufficient role: member attempting to create a draft (expected 403):")
st_forb, _, _ = http_request("POST", f"{lab.hub_url}/v1/drafts", token="member-token", body="dsl: eunomia.profiles/v1")
assert st_forb == 403

print("3) Authenticated request with member-token before publishing versions (expected 404 API_NOT_FOUND):")
st_empty, _, _ = http_request("GET", f"{lab.hub_url}/v1/versions", token="member-token")
assert st_empty == 404

1) Unauthenticated request (expected 401):
2) Insufficient role: member attempting to create a draft (expected 403):
3) Authenticated request with member-token before publishing versions (expected 404 API_NOT_FOUND):


connection,close
content-length,178
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"
www-authenticate,"Bearer realm=""vhub"""


connection,close
content-length,195
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,173
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 3. Creating Drafts and Optimistic Concurrency (`REQ-HUB-018`)

Using the **`editor`** role (`editor-token`), we create a draft by posting the `v1.0.0.yaml` bundle to `POST /v1/drafts`.
Next, we test optimistic concurrency control on `PUT /v1/drafts/{id}`:
1. Missing `If-Match` header -> **`428 Precondition Required` (`API_PRECONDITION_REQUIRED`)**.
2. Valid `If-Match: "rev-1"` -> **`200 OK`** (advances ETag to `"rev-2"`).
3. Reusing the stale `"rev-1"` ETag -> **`412 Precondition Failed` (`API_PRECONDITION_FAILED`)**.

In [4]:
draft_yaml = (lab.root / "conformance/f6/drafts/v1.0.0.yaml").read_text(encoding="utf-8")

print("1) Create draft (POST /v1/drafts):")
st, hdrs, draft_resp = http_request("POST", f"{lab.hub_url}/v1/drafts", token="editor-token", body=draft_yaml)
assert st == 201
draft_id = draft_resp["id"]
etag_v1 = hdrs["etag"]

print("2) Attempt PUT without If-Match (expected 428 API_PRECONDITION_REQUIRED):")
st_no_if, _, _ = http_request("PUT", f"{lab.hub_url}/v1/drafts/{draft_id}", token="editor-token", body=draft_yaml)
assert st_no_if == 428

print("3) Replace draft with valid If-Match (expected 200 OK):")
st_put, hdrs_put, _ = http_request(
    "PUT",
    f"{lab.hub_url}/v1/drafts/{draft_id}",
    token="editor-token",
    headers={"If-Match": etag_v1},
    body=draft_yaml,
)
assert st_put == 200
etag_v2 = hdrs_put["etag"]

print("4) Attempt PUT with stale ETag rev-1 (expected 412 API_PRECONDITION_FAILED):")
st_stale, _, _ = http_request(
    "PUT",
    f"{lab.hub_url}/v1/drafts/{draft_id}",
    token="editor-token",
    headers={"If-Match": etag_v1},
    body=draft_yaml,
)
assert st_stale == 412

1) Create draft (POST /v1/drafts):
2) Attempt PUT without If-Match (expected 428 API_PRECONDITION_REQUIRED):
3) Replace draft with valid If-Match (expected 200 OK):
4) Attempt PUT with stale ETag rev-1 (expected 412 API_PRECONDITION_FAILED):


connection,close
content-length,33
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"
etag,"""6b9537703fa7fce7-1"""
location,/v1/drafts/6b9537703fa7fce7


connection,close
content-length,197
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,33
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"
etag,"""6b9537703fa7fce7-2"""


connection,close
content-length,239
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 4. Validating (`:validate`) and Proposing a Candidate (`:propose`, `REQ-HUB-002`)

Before proposing, the editor can call `POST /v1/drafts/{id}:validate`.
When calling `POST /v1/drafts/{id}:propose`, the Hub compiles the YAML bundle with `compiler_dsl`, stores all artifacts in the content-addressed `BlobStore` by SHA-256, and freezes an **immutable Candidate**:

In [5]:
print("1) Validate draft (POST /v1/drafts/{id}:validate):")
st_val, _, _ = http_request("POST", f"{lab.hub_url}/v1/drafts/{draft_id}:validate", token="editor-token")
assert st_val == 200

print("2) Propose candidate (POST /v1/drafts/{id}:propose):")
st_prop, _, cand = http_request("POST", f"{lab.hub_url}/v1/drafts/{draft_id}:propose", token="editor-token")
assert st_prop == 201
cand_id = cand["id"]

print("3) Inspect frozen candidate (GET /v1/candidates/{id}):")
st_get_c, _, _ = http_request("GET", f"{lab.hub_url}/v1/candidates/{cand_id}", token="releaser-token")
assert st_get_c == 200

1) Validate draft (POST /v1/drafts/{id}:validate):
2) Propose candidate (POST /v1/drafts/{id}:propose):
3) Inspect frozen candidate (GET /v1/candidates/{id}):


connection,close
content-length,46
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,1103
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,1103
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 5. Role Separation and Release Approval (`:approve`, `REQ-HUB-003`, `REQ-HUB-005`)

The specification requires that an `editor` **cannot approve** a candidate (`REQ-HUB-003`): only the `authority` (`releaser`) role can execute `:approve`.
Furthermore, once a candidate is approved, attempting to approve it a second time returns `409 Conflict` (`API_INVALID_STATE`, `REQ-HUB-002`).

In [6]:
print("1) Editor attempts to approve the candidate (expected 403 API_FORBIDDEN_ROLE):")
st_forb, _, _ = http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_id}:approve", token="editor-token", body={})
assert st_forb == 403

print("2) Releaser approves the candidate (expected 201 Created with signed eunomia.version/v1 envelope):")
st_app, _, release_rec = http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_id}:approve", token="releaser-token", body={})
assert st_app == 201

print("3) Attempt to approve the same candidate a second time (expected 409 API_INVALID_STATE):")
st_dup, _, _ = http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_id}:approve", token="releaser-token", body={})
assert st_dup == 409

1) Editor attempts to approve the candidate (expected 403 API_FORBIDDEN_ROLE):
2) Releaser approves the candidate (expected 201 Created with signed eunomia.version/v1 envelope):
3) Attempt to approve the same candidate a second time (expected 409 API_INVALID_STATE):


connection,close
content-length,214
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,555
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,190
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 6. Signed Listing (`/v1/versions`) and Immutable Distribution (`/pub/*`)

Once version `1.0.0` is approved:
- `GET /v1/versions` returns the signed `eunomia.versions/v1` listing with `Cache-Control: private, no-cache` (`REQ-HUB-012`).
- `GET /pub/demo` responds `303 See Other` redirecting to the head version's manifest (`REQ-HUB-016`, `DP-15`).
- `GET /pub/packages/1.0.0/manifest.json`, `space-lock.json`, and versioned routes `/pub/demo/1.0.0/...` are served without authentication with `Cache-Control: public, max-age=31536000, immutable` and `ETag: "<sha256>"` (`REQ-SYS-005`, `REQ-HUB-011`).

In [7]:
print("1) Signed version listing GET /v1/versions:")
st_list, hdrs_list, listing = http_request("GET", f"{lab.hub_url}/v1/versions", token="member-token")
assert st_list == 200
assert "private, no-cache" in hdrs_list.get("cache-control", "")

print("2) Advance clock past the P2D propagation window (2026-09-03T12:00:00Z) and query GET /pub/demo (303 See Other):")
lab.set_clock("2026-09-03T12:00:00Z")
st_red, hdrs_red, _ = http_request("GET", f"{lab.hub_url}/pub/demo", follow_redirects=False)
assert st_red == 303

print("3) Signed package manifest GET /pub/demo/1.0.0/manifest.json (also addressable by digest at /pub/packages/{digest}/manifest.json):")
st_man, hdrs_man, manifest = http_request("GET", f"{lab.hub_url}/pub/demo/1.0.0/manifest.json")
assert st_man == 200
assert "immutable" in hdrs_man.get("cache-control", "")

print("4) Direct download of a compiled artifact (policy JSON Schema):")
st_sch, _, _ = http_request("GET", f"{lab.hub_url}/pub/demo/1.0.0/profiles/odrl/f6/1.0.0/schema.json")
assert st_sch == 200

1) Signed version listing GET /v1/versions:
2) Advance clock past the P2D propagation window (2026-09-03T12:00:00Z) and query GET /pub/demo (303 See Other):
3) Signed package manifest GET /pub/demo/1.0.0/manifest.json (also addressable by digest at /pub/packages/{digest}/manifest.json):
4) Direct download of a compiled artifact (policy JSON Schema):


cache-control,"private, no-cache"
connection,close
content-length,609
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


🕒 Clock updated → 2026-09-03T12:00:00Z

cache-control,no-cache
connection,close
content-length,0
date,"Thu, 03 Sep 2026 12:00:00 GMT"
location,/pub/demo/1.0.0/manifest.json


cache-control,"public, max-age=31536000, immutable"
connection,close
content-length,2032
content-type,application/json
date,"Thu, 03 Sep 2026 12:00:00 GMT"
etag,"""9109e5b8ca1178902795040ad016be38a6a978c4c6fbb6e21492ac5e78f6e3ec"""


cache-control,"public, max-age=31536000, immutable"
connection,close
content-type,application/schema+json
date,"Thu, 03 Sep 2026 12:00:00 GMT"
etag,"""764b9f692a51e18ff079adf26fb8c902dc3caf5d529e30f033a579b478f4de80"""
transfer-encoding,chunked


## 7. Immutable Governance Event Log (`GET /v1/events`)

Every mutation on the Hub is sequentially recorded in the immutable SQLite `events` table (`REQ-HUB-020`):

In [8]:
st_ev, _, events_resp = http_request("GET", f"{lab.hub_url}/v1/events?after=0", token="editor-token")
assert st_ev == 200
lab.close()

connection,close
content-length,685
content-type,application/json
date,"Thu, 03 Sep 2026 12:00:00 GMT"
